# HeadSwap production pipeline

Single production path only — automatic dark-lighting routing and optional Magic Hour face detection.

**Routing when `ENABLE_LIGHTING_ROUTE=True`:**
- single person → existing `crop_stitch`
- multi-person, normal lighting → production `crop_stitch` (r64, ref_boost 3.5)
- multi-person, dark lighting → `full_frame` (full v1.2, ref_boost 4)

**Usage:**
1. Runtime → **Restart session**
2. Runtime → **Run all**
3. Photos loaded/uploaded in §3; unified production result produced in §4 and displayed in §5.

## 1 · Settings


In [ ]:
# === User-facing knobs ===
SEED = 46
STEPS = 8
CFG = 1.0
OUTPUT_LONG_SIDE = 1024
DEBUG = False

# Head selection on multi-person body (0 = auto / policy)
TARGET_HEAD = 0
BODY_FACE_POLICY = "largest"

# Automatic dark-lighting router
ENABLE_LIGHTING_ROUTE = True
DARK_LIGHTING_THRESHOLD = 110.0

# Face-localized identity attention boost (ref_boost_mask) on full-frame path
FULL_FRAME_REF_BOOST_MASK = True

# Face detection backend: "current" (InsightFace/OpenCV, default) | "magic_hour"
FACE_DETECTION_BACKEND = "current"
MAGIC_HOUR_API_KEY = ""  # optional here; loaded automatically from .env or Colab Secret

# Upload your own photos in §3 (set True to use built-in demo pair instead)
USE_DEMO_PAIR = False
DEMO_BODY = "data/demo/body_multi.png"
DEMO_FACE = "data/demo/face.png"

REPO_BRANCH = "feature/lighting-route-and-magichour"
PINNED_COMMIT = None
USE_DRIVE = True

print(f"Settings — branch={REPO_BRANCH}")
print(f"  FACE_DETECTION_BACKEND={FACE_DETECTION_BACKEND}")
print(f"  ENABLE_LIGHTING_ROUTE={ENABLE_LIGHTING_ROUTE} (threshold={DARK_LIGHTING_THRESHOLD})")
print(f"  FULL_FRAME_REF_BOOST_MASK={FULL_FRAME_REF_BOOST_MASK}")
print(f"  TARGET_HEAD={TARGET_HEAD or 'auto (' + BODY_FACE_POLICY + ')'}")
print(f"  USE_DEMO_PAIR={USE_DEMO_PAIR}")
print(f"  REPO_BRANCH={REPO_BRANCH}  PINNED={PINNED_COMMIT or 'HEAD'}")
print(f"  USE_DRIVE={USE_DRIVE}")


## 2 · Setup (GPU · Drive · repo · models)


In [ ]:
#@title Setup
from pathlib import Path
import importlib.util
import os
import subprocess

assert Path("/content").exists(), "Open this notebook in Google Colab."

import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime → Change runtime type → GPU (A100 preferred), then Run all.")
print(f"✓ GPU  {torch.cuda.get_device_name(0)}")

USE_DRIVE = bool(globals().get("USE_DRIVE", True))
DRIVE_OK = False
if USE_DRIVE:
    try:
        from google.colab import drive
        print("→ Mounting Drive (optional)…")
        drive.mount("/content/drive", force_remount=False)
        DRIVE_OK = Path("/content/drive/MyDrive").exists()
        print("✓ Drive mounted" if DRIVE_OK else "⚠ Drive path missing after mount")
    except Exception as exc:
        DRIVE_OK = False
        print(f"⚠ Drive mount failed ({type(exc).__name__}: {exc})")
        print("  Continuing without Drive — models go to /content/models")
else:
    print("USE_DRIVE=False — skipping Drive mount")

REPO_URL = "https://github.com/malihashar/headswap_V2.git"
REPO = Path("/content/headswap_V2")
REPO_BRANCH = globals().get("REPO_BRANCH") or "feature/lighting-route-and-magichour"
print("→ Syncing repo…")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=False)
subprocess.run(["git", "-C", str(REPO), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=False)
subprocess.run(
    ["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH],
    check=False,
)
pin = globals().get("PINNED_COMMIT")
if pin:
    print(f"→ Pinning commit {pin}…")
    subprocess.run(["git", "-C", str(REPO), "checkout", str(pin)], check=False)
os.chdir(REPO)
print("✓", subprocess.getoutput(f"git -C {REPO} rev-parse --abbrev-ref HEAD"),
      subprocess.getoutput(f"git -C {REPO} rev-parse --short HEAD"))

!pip install -q -e .

spec = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
colab_env = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_env)
PATHS = colab_env.apply_env(colab_env.default_paths(use_drive=DRIVE_OK))
print(f"  model_store → {PATHS['model_store']}")

spec_d = importlib.util.spec_from_file_location("colab_demo", REPO / "scripts" / "colab_demo.py")
colab_demo = importlib.util.module_from_spec(spec_d)
spec_d.loader.exec_module(colab_demo)

pin = globals().get("PINNED_COMMIT")
VERSIONS = colab_demo.collect_versions(repo=REPO, comfyui=PATHS["comfyui"], pinned_commit=pin)

print("→ Installing ComfyUI + Krea2 nodes…")
!bash scripts/setup_colab.sh
!bash scripts/setup_krea2_nodes.sh

print("→ Downloading / verifying models…")
_dl = subprocess.run(
    [
        "python", "scripts/download_krea2.py",
        "--comfy", os.environ["COMFYUI_PATH"],
        "--store-dir", os.environ["HEADSWAP_MODEL_STORE"],
        "--staging-dir", os.environ["HEADSWAP_STAGING_DIR"],
        "--backend", "auto",
        "--disable-xet",
        "--include-optional",
    ],
    check=False,
)
if _dl.returncode != 0:
    raise SystemExit(f"Model download failed (exit={_dl.returncode}). Re-run §2.")

colab_demo.verify_models(PATHS["model_store"], check_sizes=True)
VERSIONS = colab_demo.collect_versions(repo=REPO, comfyui=PATHS["comfyui"])
colab_demo.print_versions(VERSIONS)

# Load MAGIC_HOUR_API_KEY from Colab Secrets (always, so §3b face selector works)
_mh_key_explicit = globals().get("MAGIC_HOUR_API_KEY") or ""
if str(_mh_key_explicit).strip():
    import os; os.environ["MAGIC_HOUR_API_KEY"] = str(_mh_key_explicit).strip()
    print("\u2713 MAGIC_HOUR_API_KEY set from §1 variable")
else:
    try:
        from google.colab import userdata
        import os
        _mh_secret = userdata.get("MAGIC_HOUR_API_KEY")
        if _mh_secret:
            os.environ["MAGIC_HOUR_API_KEY"] = str(_mh_secret).strip()
            print("\u2713 MAGIC_HOUR_API_KEY loaded from Colab Secrets")
        else:
            print("\u26a0 MAGIC_HOUR_API_KEY secret not found \u2014 face selector will use InsightFace")
    except Exception as _mh_exc:
        print(f"\u26a0 Could not load MAGIC_HOUR_API_KEY: {_mh_exc}")

colab_demo.ok("Setup ready")
print("If FIRST custom-node install on a fresh runtime: Runtime → Restart session, then Run all from §1.")


## 3 · Inputs (demo pair or upload)


In [ ]:
# §3 Inputs — demo pair or upload
import importlib.util
import shutil
import subprocess
import urllib.request
from pathlib import Path

from IPython.display import display, Markdown
from PIL import Image

REPO = Path("/content/headswap_V2")
assert REPO.is_dir(), "Repo missing — run §2 Setup first."

BRANCH = str(globals().get("REPO_BRANCH") or "feature/lighting-route-and-magichour")
PIN = globals().get("PINNED_COMMIT")

print(f"→ Syncing {BRANCH} …")
subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=False)
subprocess.run(
    ["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"],
    check=False,
)
subprocess.run(
    ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
    check=False,
)
if PIN:
    subprocess.run(["git", "-C", str(REPO), "checkout", str(PIN)], check=False)
print(
    "✓ repo",
    subprocess.getoutput(f"git -C {REPO} rev-parse --short HEAD"),
    subprocess.getoutput(f"git -C {REPO} rev-parse --abbrev-ref HEAD"),

# Purge cached headswap modules so fresh code from git is imported
import sys as _sys
for _mod in list(_sys.modules.keys()):
    if _mod.startswith('headswap'):
        del _sys.modules[_mod]
print('✓ Cleared in-memory Python module cache for headswap')
)

spec = importlib.util.spec_from_file_location("colab_demo", REPO / "scripts" / "colab_demo.py")
colab_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_demo)

custom = REPO / "data" / "custom"
custom.mkdir(parents=True, exist_ok=True)
BODY_PATH = custom / "body.png"
FACE_PATH = custom / "face.png"
CACHE = REPO / ".cache" / "headswap_v2"
CACHE.mkdir(parents=True, exist_ok=True)

USE_DEMO_PAIR = bool(globals().get("USE_DEMO_PAIR", False))
DEMO_BODY = str(globals().get("DEMO_BODY", "data/demo/body_multi.png"))
DEMO_FACE = str(globals().get("DEMO_FACE", "data/demo/face.png"))

def _ensure_demo(rel: str, dest: Path) -> Path:
    """Copy from repo, or download from GitHub raw if the file is missing."""
    src = REPO / rel
    if src.is_file() and src.stat().st_size > 1000:
        shutil.copy2(src, dest)
        print(f"✓ copied {rel} → {dest.name} ({dest.stat().st_size} bytes)")
        return dest
    refs = [PIN, BRANCH, "main"]
    refs = [str(r) for r in refs if r]
    last_err = None
    dest.parent.mkdir(parents=True, exist_ok=True)
    for ref in refs:
        url = f"https://raw.githubusercontent.com/malihashar/headswap_V2/{ref}/{rel}"
        print(f"→ local missing; downloading {url}")
        try:
            urllib.request.urlretrieve(url, dest)
            if dest.is_file() and dest.stat().st_size > 1000:
                print(f"✓ downloaded → {dest} ({dest.stat().st_size} bytes)")
                return dest
        except Exception as exc:
            last_err = exc
            print(f"  failed: {exc}")
    raise SystemExit(
        f"Could not fetch demo image {rel}. last_error={last_err}\n"
        "Runtime → Restart session, then Run all from §1."
    )

if USE_DEMO_PAIR:
    print("Using demo pair (no upload dialog).")
    _ensure_demo(DEMO_BODY, BODY_PATH)
    _ensure_demo(DEMO_FACE, FACE_PATH)
else:
    from google.colab import files

    print("── Upload 1 of 2: BODY photo (scene/group photo to edit) ──")
    up_body = files.upload()
    if not up_body:
        raise SystemExit("No body photo uploaded. Re-run this cell and upload a body photo.")
    colab_demo.save_upload(next(iter(up_body.values())), BODY_PATH)
    print(f"✓ Body saved ({BODY_PATH.stat().st_size} bytes)")
    print()
    print("── Upload 2 of 2: FACE photo (identity donor face to swap in) ──")
    up_face = files.upload()
    if not up_face:
        raise SystemExit("No face photo uploaded. Re-run this cell and upload a face photo.")
    colab_demo.save_upload(next(iter(up_face.values())), FACE_PATH)
    print(f"✓ Face saved ({FACE_PATH.stat().st_size} bytes)")

body_im = Image.open(BODY_PATH).convert("RGB")
face_im = Image.open(FACE_PATH).convert("RGB")
print(f"Loaded body={body_im.size} face={face_im.size}")

try:
    body_face = colab_demo.require_face(body_im, CACHE, "body")
    face_face = colab_demo.require_face(face_im, CACHE, "face")
except colab_demo.DemoError as exc:
    raise SystemExit(
        f"{exc}\n\n"
        "You must use *real* photos with visible faces.\n"
        "If you still see this after a pull: Runtime → Restart session, Run all.\n"
        "Or set USE_DEMO_PAIR=False and upload your own photos."
    ) from exc

display(Markdown("### Inputs ready"))
print(f"Body: {body_im.size}, faces={body_face['face_count']}")
display(body_im)
print(f"Face: {face_im.size}, faces={face_face['face_count']}")
display(face_im)
colab_demo.ok(f"Saved → {BODY_PATH} , {FACE_PATH}")


## 3b · Face selector (choose which face to swap)


In [ ]:
# §3b Face selector — detect faces in the body photo, show left-to-right,
# let user pick which one to swap (or auto-select if only 1 face).
import os
import io
import sys
import math
import tempfile
import urllib.request
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, Markdown

REPO = Path("/content/headswap_V2")
BODY_PATH = REPO / "data" / "custom" / "body.png"
assert BODY_PATH.is_file(), "Run §3 first to upload the body photo."

body_im = Image.open(BODY_PATH).convert("RGB")

# ── Try Magic Hour face detection if API key is available ──
MH_KEY = os.environ.get("MAGIC_HOUR_API_KEY", "").strip()
BACKEND = str(globals().get("FACE_DETECTION_BACKEND", "current")).strip().lower()

detected_faces = []  # list of dicts: {index, label, bbox_xywh, thumb_im}
_mh_precomputed = None

sys.path.insert(0, str(REPO / "src"))

if MH_KEY and BACKEND in ("magic_hour", "magichour", "mh"):
    try:
        from headswap.magichour.face_detection import detect_faces_magic_hour, MagicHourFaceDetectionClient
        print("\u2192 Running Magic Hour face detection...")
        client = MagicHourFaceDetectionClient(api_key=MH_KEY)
        import tempfile
        fd, tmp_name = tempfile.mkstemp(prefix="mh_body_", suffix=".png")
        os.close(fd)
        body_im.save(tmp_name, format="PNG")
        mh_result = client.detect_local_image(tmp_name)
        os.unlink(tmp_name)
        print(f"\u2713 Magic Hour: {mh_result.face_count} face(s) detected (job={mh_result.id})")
        _mh_precomputed = {
            "backend": "magic_hour",
            "id": mh_result.id,
            "status": mh_result.status,
            "face_count": mh_result.face_count,
            "faces": [{"path": f.path, "url": f.url} for f in mh_result.faces],
            "credits_charged": mh_result.credits_charged,
        }
        # Download face crop thumbnails from MH urls & sort left-to-right
        # MH doesn't return bboxes — re-detect each crop in original scene
        # via InsightFace for left-to-right ordering.
        from headswap.preprocess import detect_faces
        import numpy as np
        body_np = np.array(body_im)
        scene_faces = detect_faces(body_np, REPO / '.cache' / 'headswap_v2')  # returns list of FaceBox sorted by confidence
        # sort scene_faces by x1 (left-to-right)
        scene_faces_sorted = sorted(scene_faces, key=lambda f: f.x0)
        for idx, sf in enumerate(scene_faces_sorted):
            x1, y1, x2, y2 = int(sf.x0), int(sf.y0), int(sf.x1), int(sf.y1)
            pad = max(10, int((x2 - x1) * 0.15))
            x1c = max(0, x1 - pad); y1c = max(0, y1 - pad)
            x2c = min(body_im.width, x2 + pad); y2c = min(body_im.height, y2 + pad)
            crop = body_im.crop((x1c, y1c, x2c, y2c)).resize((160, 160))
            detected_faces.append({"index": idx + 1, "label": f"Face {idx+1}",
                                    "bbox": (x1, y1, x2, y2), "thumb": crop})
        print(f"\u2713 Sorted {len(detected_faces)} face(s) left-to-right from scene")
    except Exception as _mh_e:
        print(f"\u26a0 Magic Hour detection failed ({type(_mh_e).__name__}: {_mh_e}); falling back to InsightFace")
        detected_faces = []

if not detected_faces:
    # Fallback: InsightFace left-to-right
    try:
        from headswap.preprocess import detect_faces
        import numpy as np
        body_np = np.array(body_im)
        scene_faces = detect_faces(body_np, REPO / '.cache' / 'headswap_v2')
        scene_faces_sorted = sorted(scene_faces, key=lambda f: f.x0)
        for idx, sf in enumerate(scene_faces_sorted):
            x1, y1, x2, y2 = int(sf.x0), int(sf.y0), int(sf.x1), int(sf.y1)
            pad = max(10, int((x2 - x1) * 0.15))
            x1c = max(0, x1 - pad); y1c = max(0, y1 - pad)
            x2c = min(body_im.width, x2 + pad); y2c = min(body_im.height, y2 + pad)
            crop = body_im.crop((x1c, y1c, x2c, y2c)).resize((160, 160))
            detected_faces.append({"index": idx + 1, "label": f"Face {idx+1}",
                                    "bbox": (x1, y1, x2, y2), "thumb": crop})
        print(f"\u2713 InsightFace: {len(detected_faces)} face(s) detected (left-to-right)")
    except Exception as _if_e:
        print(f"\u26a0 InsightFace detection failed: {_if_e}")

if not detected_faces:
    print("\u26a0 No faces detected. TARGET_HEAD left as-is from §1.")
elif len(detected_faces) == 1:
    TARGET_HEAD = 1
    print(f"\u2713 Single face detected — auto-selecting Face 1. TARGET_HEAD={TARGET_HEAD}")
    if _mh_precomputed:
        globals()["_MH_PRECOMPUTED"] = _mh_precomputed
    display(detected_faces[0]["thumb"])
else:
    # Build side-by-side thumbnail strip with labels
    n = len(detected_faces)
    THUMB_W, THUMB_H, LABEL_H = 160, 160, 24
    GAP = 8
    strip_w = n * THUMB_W + (n - 1) * GAP
    strip_h = THUMB_H + LABEL_H
    strip = Image.new("RGB", (strip_w, strip_h), (30, 30, 30))
    draw = ImageDraw.Draw(strip)
    for i, fd in enumerate(detected_faces):
        x_off = i * (THUMB_W + GAP)
        strip.paste(fd["thumb"], (x_off, 0))
        draw.rectangle([x_off, THUMB_H, x_off + THUMB_W, THUMB_H + LABEL_H], fill=(30, 30, 30))
        label_text = fd["label"]
        text_x = x_off + THUMB_W // 2 - len(label_text) * 4
        draw.text((text_x, THUMB_H + 4), label_text, fill=(255, 255, 255))
    display(Markdown("### Select the face to swap in (left to right: Face 1 = leftmost)"))
    display(strip)
    print("\nFaces detected (left to right):")
    for fd in detected_faces:
        x1, y1, x2, y2 = fd["bbox"]
        print(f"  Face {fd['index']}: bbox=({x1},{y1},{x2},{y2})")
    print()
    # ipywidgets radio buttons selector
    try:
        import ipywidgets as widgets
        options = [(f"Face {fd['index']} (leftmost)" if fd['index'] == 1 else f"Face {fd['index']}", fd['index'])
                   for fd in detected_faces]
        radio = widgets.RadioButtons(options=options, value=options[0][1],
                                     description="Target face:", style={"description_width": "initial"})
        out = widgets.Output()
        global TARGET_HEAD
        TARGET_HEAD = radio.value
        if _mh_precomputed:
            globals()["_MH_PRECOMPUTED"] = _mh_precomputed
        with out:
            print(f"\u2713 TARGET_HEAD set to {TARGET_HEAD}")
        def on_change(change):
            global TARGET_HEAD
            TARGET_HEAD = change["new"]
            if _mh_precomputed:
                globals()["_MH_PRECOMPUTED"] = _mh_precomputed
            with out:
                out.clear_output()
                print(f"\u2713 TARGET_HEAD set to {TARGET_HEAD}")
        radio.observe(on_change, names="value")
        display(radio)
        display(out)
        print("Click a bullet point above to select a face, then run §4.")
    except ImportError:
        # No ipywidgets — fall back to plain input
        choice = input(f"Enter face number (1-{len(detected_faces)}, left=1): ").strip()
        TARGET_HEAD = int(choice) if choice.isdigit() else 1
        print(f"\u2713 TARGET_HEAD={TARGET_HEAD}")
        if _mh_precomputed:
            globals()["_MH_PRECOMPUTED"] = _mh_precomputed


## 4 · Run production pipeline


In [ ]:
# §4 Run production pipeline (single path with automatic routing)
import json
import time
import sys
from datetime import datetime, timezone
from pathlib import Path

from IPython.display import display, Markdown
from PIL import Image

REPO = Path("/content/headswap_V2")
assert REPO.is_dir(), "Repo missing — run §2 first."

spec_env = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
colab_env = importlib.util.module_from_spec(spec_env)
spec_env.loader.exec_module(colab_env)
PATHS = colab_env.apply_env(colab_env.default_paths(use_drive=bool(globals().get('DRIVE_OK', False))))
colab_env.ensure_import_path(REPO)

spec = importlib.util.spec_from_file_location("colab_demo", REPO / "scripts" / "colab_demo.py")
colab_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_demo)

BODY_PATH = REPO / "data" / "custom" / "body.png"
FACE_PATH = REPO / "data" / "custom" / "face.png"
if not BODY_PATH.is_file() or not FACE_PATH.is_file():
    raise SystemExit(
        f"Missing inputs:\n  {BODY_PATH.exists()=} {BODY_PATH}\n  {FACE_PATH.exists()=} {FACE_PATH}\n"
        "Run §3 first (demo pair or upload)."
    )

# Force fresh module reload from disk
import sys as _sys
for _mod in list(_sys.modules.keys()):
    if _mod.startswith('headswap'):
        del _sys.modules[_mod]
from headswap.config import load_config
from headswap.pipelines import create_pipeline
from headswap.pipelines.krea2 import get_shared_krea2_runtime

t_start = time.perf_counter()

cfg = load_config(REPO / "configs" / "krea2_identity_edit.yaml")
backend = str(globals().get("FACE_DETECTION_BACKEND", "current")).strip().lower()
cfg.update({
    "seed": int(globals().get("SEED", 46)),
    "steps": int(globals().get("STEPS", 8)),
    "cfg": float(globals().get("CFG", 1.0)),
    "max_body_dim": int(globals().get("OUTPUT_LONG_SIDE", 1024)),
    "max_dim": int(globals().get("OUTPUT_LONG_SIDE", 1024)),
    "save_debug": bool(globals().get("DEBUG", False)),
    "verbose": bool(globals().get("DEBUG", False)),
    "enable_lighting_route": bool(globals().get("ENABLE_LIGHTING_ROUTE", True)),
    "dark_lighting_threshold": float(globals().get("DARK_LIGHTING_THRESHOLD", 110.0)),
    "face_detection_backend": backend,
    "body_face_policy": str(globals().get("BODY_FACE_POLICY", "largest")),
    "body_face_index": max(0, int(globals().get("TARGET_HEAD", 0)) - 1) if int(globals().get("TARGET_HEAD", 0)) >= 1 else 0,
    "enable_multi_face_features": False,
    "full_frame_ref_boost_mask": bool(globals().get("FULL_FRAME_REF_BOOST_MASK", True)),
    "face_swap_mode": "single",
    "mask_crop_stitch": True,
})


# Pick up TARGET_HEAD if updated by §3b face selector
if globals().get("TARGET_HEAD"):
    cfg["body_face_policy"] = "index"
    cfg["body_face_index"] = max(0, int(globals()["TARGET_HEAD"]) - 1)
# Pass MH precomputed result if §3b ran MH detection
if globals().get("_MH_PRECOMPUTED"):
    cfg["magic_hour_precomputed_result"] = globals()["_MH_PRECOMPUTED"]

target_head = int(globals().get("TARGET_HEAD", 0))
if target_head >= 1:
    cfg["body_face_policy"] = "index"
    cfg["body_face_index"] = target_head - 1

body = Image.open(BODY_PATH).convert("RGB")
face = Image.open(FACE_PATH).convert("RGB")

run_stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
RUN_DIR = REPO / "results" / "production_runs" / f"prod_{run_stamp}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f"→ Running production pipeline…")
print(f"  backend={backend}  lighting_route={cfg['enable_lighting_route']}  threshold={cfg['dark_lighting_threshold']}")
print(f"  body={BODY_PATH}  face={FACE_PATH}")
print(f"  out={RUN_DIR}")
print("  (loading models on first run — expect several minutes)")

try:
    # Ensure ComfyUI is on sys.path so `import server` works
    import sys as _sys
    _comfy_path = str(PATHS.get("comfyui", "/content/ComfyUI"))
    if _comfy_path not in _sys.path:
        _sys.path.insert(0, _comfy_path)
    runtime = get_shared_krea2_runtime(init_custom_nodes=True)
    pipeline = create_pipeline(cfg, runtime=runtime)
    result = pipeline.run(body, face, out_dir=RUN_DIR / "debug")

    RESULT_PATH = RUN_DIR / "final_output.png"
    result.image.save(RESULT_PATH)

    meta = dict(result.meta or {})
    route_meta = meta.get("lighting_route") or {}
    mh_meta = meta.get("magic_hour_face_detection") or {}

    wall_s = round(time.perf_counter() - t_start, 2)
    meta["wall_s"] = wall_s
    meta["latency_s"] = round(result.latency_s, 2)

    META_PATH = RUN_DIR / "meta.json"
    META_PATH.write_text(json.dumps(meta, indent=2, default=str), encoding="utf-8")

    report_lines = [
        "# Production Pipeline Run Report",
        "",
        f"- **Timestamp**: `{run_stamp}`",
        f"- **Latency**: `{meta['latency_s']}s` (wall: `{wall_s}s`)",
        f"- **Face Detection Backend**: `{backend}`",
        f"- **Lighting Route Enabled**: `{route_meta.get('enable_lighting_route')}`",
        f"- **Effective Edit Mode**: `{meta.get('edit_mode')}`",
        f"- **Route Chosen**: `{route_meta.get('route')}` (`reason: {route_meta.get('reason')}`)",
        f"- **Faces Detected**: `{route_meta.get('faces_detected', meta.get('body_face_count'))}`",
        f"- **Lighting Metric (HSV-V)**: `{route_meta.get('lighting_metric', 'N/A')}` (threshold: `{route_meta.get('dark_lighting_threshold', 'N/A')}`)",
        f"- **Is Dark**: `{route_meta.get('is_dark', 'N/A')}`",
        f"- **LoRA Loaded**: `{meta.get('loras_loaded')}`",
        f"- **Ref Boost**: `{meta.get('ref_boost')}`",
        "",
    ]
    if mh_meta:
        report_lines.extend([
            "## Magic Hour Face Detection",
            f"- **Status**: `{mh_meta.get('status')}`",
            f"- **Face Count**: `{mh_meta.get('face_count')}`",
            f"- **Credits Charged**: `{mh_meta.get('credits_charged', 0)}`",
            "",
        ])

    REPORT_PATH = RUN_DIR / "REPORT.md"
    REPORT_PATH.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

    RUN_OK = True
    RUN_ERROR = None
    print(f"\n✓ Production pipeline finished in {wall_s}s")
    print(f"  REPORT → {REPORT_PATH}")
    print(f"  output → {RESULT_PATH}")

except Exception as exc:
    RUN_OK = False
    RUN_ERROR = f"{type(exc).__name__}: {exc}"
    print(f"\n❌ Pipeline execution failed: {RUN_ERROR}")
    raise


## 5 · Results


In [ ]:
# §5 Results display & download
from pathlib import Path
from IPython.display import display, Markdown
from PIL import Image
from google.colab import files

REPO = Path("/content/headswap_V2")
run_dir = Path(globals().get("RUN_DIR") or (REPO / "results" / "production_runs"))

if not globals().get("RUN_OK"):
    raise SystemExit(globals().get("RUN_ERROR") or "§4 did not succeed — scroll up for the error.")

display(Markdown("### Production Pipeline Output"))

final_img = Path(globals().get("RESULT_PATH") or (run_dir / "final_output.png"))
if final_img.is_file():
    display(Markdown(f"**Final Output Image** (`{final_img.name}`)"))
    display(Image.open(final_img))
else:
    print("No final_output.png found under", run_dir)

report_file = Path(globals().get("REPORT_PATH") or (run_dir / "REPORT.md"))
if report_file.is_file():
    display(Markdown("### REPORT.md"))
    display(Markdown(report_file.read_text(encoding="utf-8")))
    try:
        # report download disabled per user preference
        pass
    except Exception as exc:
        print("report download skipped:", exc)

if final_img.is_file():
    try:
        files.download(str(final_img))
    except Exception as exc:
        print("image download skipped:", exc)
